In [ ]:
# 1. Install arc-agi runtime from offline competition wheelhouse
import os, sys, glob, subprocess

print("Installing arc-agi runtime from offline wheelhouse...")
wheel_dirs = glob.glob("/kaggle/input/**/arc_agi_3_wheels", recursive=True)
if wheel_dirs:
    wheel_dir = wheel_dirs[0]
    print(f"Found wheels directory: {wheel_dir}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", wheel_dir, "arc-agi", "python-dotenv", "pillow"], check=False)
else:
    print("arc_agi_3_wheels not found in /kaggle/input; dependencies should be pre-installed.")


In [ ]:
# 2. Cohezion ARC-AGI-3 Directed Rarity-Search Agent (v15)
import hashlib
import json
import logging
import os
import random
import sys
import time
from collections import deque
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Set
import numpy as np

import arc_agi
import arcengine
from arcengine import GameAction, GameState, FrameDataRaw

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("CohezionARC3")

def hash_frame(frame_layers: list) -> str:
    if not frame_layers:
        return ""
    return hashlib.md5(np.ascontiguousarray(frame_layers[0]).tobytes()).hexdigest()[:16]

def find_components(grid: np.ndarray) -> list[dict[str, Any]]:
    """Extract foreground connected components sorted by color rarity ascending, then size ascending."""
    h, w = grid.shape
    visited = np.zeros((h, w), dtype=bool)
    components = []
    vals, counts = np.unique(grid, return_counts=True)
    bg_color = vals[np.argmax(counts)]
    color_freq = dict(zip(vals, counts))

    for r in range(h):
        for c in range(w):
            if visited[r, c] or grid[r, c] == bg_color:
                continue
            color = int(grid[r, c])
            q = deque([(r, c)])
            visited[r, c] = True
            pixels = []
            while q:
                cr, cc = q.popleft()
                pixels.append((cr, cc))
                for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nr, nc = cr + dr, cc + dc
                    if 0 <= nr < h and 0 <= nc < w and not visited[nr, nc] and grid[nr, nc] == color:
                        visited[nr, nc] = True
                        q.append((nr, nc))
            rs = [p[0] for p in pixels]
            cs = [p[1] for p in pixels]
            components.append({
                "color": color,
                "color_count": int(color_freq[color]),
                "size": len(pixels),
                "center_r": int(sum(rs) / len(rs)),
                "center_c": int(sum(cs) / len(cs)),
                "bbox": (min(rs), min(cs), max(rs), max(cs)),
            })

    # Strict Action Economy: Target rarest single-pixel and small-sprite affordances (key, player, door) first!
    components.sort(key=lambda c: (c["color_count"], c["size"]))
    return components

class CohezionDirectedRarityAgent:
    """Affordance-targeted search agent optimizing quadratic action economy ((human/ai)^2 score)."""
    def __init__(self, game_id: str, seed: int = 42):
        self.game_id = game_id
        self.rng = random.Random(seed + hash(game_id) % 100000)
        self.transition_model: dict[str, dict[int, str]] = {}
        self.state_visits: dict[str, int] = {}
        self.clicked_targets: dict[str, set[tuple[int, int]]] = {}
        self.last_state_hash: Optional[str] = None
        self.last_action_id: Optional[int] = None
        self.last_frame: Optional[np.ndarray] = None
        self.player_pos: Optional[tuple[int, int]] = None
        self.step_count = 0
        self.consecutive_loops = 0

    def choose_action(self, obs: FrameDataRaw) -> tuple[GameAction, Optional[dict[str, Any]]]:
        self.step_count += 1
        curr_hash = hash_frame(obs.frame)
        self.state_visits[curr_hash] = self.state_visits.get(curr_hash, 0) + 1
        grid = obs.frame[0] if obs.frame else None

        # Track avatar position from frame delta on directional movement
        if self.last_frame is not None and grid is not None and self.last_action_id in (1, 2, 3, 4):
            diff = (grid != self.last_frame)
            if 0 < np.sum(diff) <= 64:
                coords = np.argwhere(diff)
                self.player_pos = (int(np.mean(coords[:, 0])), int(np.mean(coords[:, 1])))

        self.last_frame = grid.copy() if grid is not None else None

        # Track state transitions and consecutive self-loops (wall collisions)
        if self.last_state_hash is not None and self.last_action_id is not None:
            if self.last_state_hash not in self.transition_model:
                self.transition_model[self.last_state_hash] = {}
            self.transition_model[self.last_state_hash][self.last_action_id] = curr_hash
            if curr_hash == self.last_state_hash:
                self.consecutive_loops += 1
            else:
                self.consecutive_loops = 0

        avail = obs.available_actions
        simple_ids = [a for a in avail if a not in (0, 6, 7)]
        has_click = 6 in avail

        if curr_hash not in self.clicked_targets:
            self.clicked_targets[curr_hash] = set()

        comps = find_components(grid) if grid is not None else []

        # 1. Click handling (Preserve v14 proven logic + jitter fallback)
        if has_click and (not simple_ids or self.consecutive_loops >= 2 or self.rng.random() < 0.40):
            unclicked = [c for c in comps if (c["center_c"], c["center_r"]) not in self.clicked_targets[curr_hash]]
            if unclicked:
                target = unclicked[0]
                coord = (target["center_c"], target["center_r"])
                self.clicked_targets[curr_hash].add(coord)
                self.last_state_hash = curr_hash
                self.last_action_id = 6
                return GameAction.ACTION6, {"x": coord[0], "y": coord[1]}
            elif comps and not simple_ids:
                # Cycle click positions within bounding boxes to prevent lock
                target = self.rng.choice(comps)
                r_min, c_min, r_max, c_max = target["bbox"]
                coord = (self.rng.randint(c_min, c_max), self.rng.randint(r_min, r_max))
                self.last_state_hash = curr_hash
                self.last_action_id = 6
                return GameAction.ACTION6, {"x": coord[0], "y": coord[1]}

        # 2. Directed Movement handling (Direct vector steering toward rarest affordance)
        if simple_ids:
            known = self.transition_model.get(curr_hash, {})
            if self.player_pos is not None and comps:
                pr, pc = self.player_pos
                target = comps[0]
                tr, tc = target["center_r"], target["center_c"]
                dr, dc = tr - pr, tc - pc
                if abs(dr) > abs(dc):
                    preferred = [1 if dr < 0 else 2, 3 if dc < 0 else 4]
                else:
                    preferred = [3 if dc < 0 else 4, 1 if dr < 0 else 2]
                for act_id in preferred:
                    if act_id in simple_ids and known.get(act_id) != curr_hash:
                        self.last_state_hash = curr_hash
                        self.last_action_id = act_id
                        return GameAction.from_id(act_id), None

            untried = [a for a in simple_ids if a not in known]
            if untried:
                chosen_id = untried[0]
            else:
                non_loop = [a for a in simple_ids if known.get(a) != curr_hash]
                if non_loop:
                    chosen_id = min(non_loop, key=lambda a: self.state_visits.get(known.get(a, ""), 0))
                else:
                    chosen_id = simple_ids[0]

            self.last_state_hash = curr_hash
            self.last_action_id = chosen_id
            return GameAction.from_id(chosen_id), None

        fallback_id = avail[0] if avail else 0
        return GameAction.from_id(fallback_id), None

# Backward compatibility alias
CohezionRaritySearchAgent = CohezionDirectedRarityAgent
CohezionGoExploreAgent = CohezionDirectedRarityAgent


In [ ]:
# 3. Competition Arcade Master Orchestrator
import os, sys, glob, time, urllib.request
from pathlib import Path

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
print(f"Cohezion ARC-AGI-3 Master Orchestrator: TRUE_SUBMISSION={TRUE_SUBMISSION}")

GATEWAY_URL = os.environ.get("ARC_BASE_URL", "http://gateway:8001/")
MAX_TOTAL_RUNTIME_S = 8.5 * 3600
START_TIME = time.time()

def wait_for_gateway(base_url: str, max_wait_s: int = 300) -> bool:
    print(f"Polling gateway at {base_url}api/games (up to {max_wait_s}s)...", flush=True)
    deadline = time.time() + max_wait_s
    while time.time() < deadline:
        try:
            req = urllib.request.Request(f"{base_url}api/games")
            with urllib.request.urlopen(req, timeout=5) as resp:
                if resp.status == 200:
                    print("✓ Gateway is online and ready!", flush=True)
                    return True
        except Exception:
            pass
        time.sleep(3)
    print("Warning: Gateway polling timed out.", flush=True)
    return False

if TRUE_SUBMISSION:
    wait_for_gateway(GATEWAY_URL)
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=GATEWAY_URL,
        arc_api_key="test-key-123",
    )
else:
    env_dirs = glob.glob("/kaggle/input/**/environment_files", recursive=True)
    env_dir = env_dirs[0] if env_dirs else "data/arc_prize/environment_files"
    print(f"Running in OFFLINE mode using environments from: {env_dir}")
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=env_dir,
    )

card_id = arcade.create_scorecard()
print(f"Opened Scorecard: {card_id}")

envs = arcade.get_environments()
print(f"Discovered {len(envs)} environments to solve.")

# In offline commit mode, run 2 environments quickly to pass the commit gate
if not TRUE_SUBMISSION:
    envs = envs[:2]

for idx, env_info in enumerate(envs):
    elapsed = time.time() - START_TIME
    if elapsed > MAX_TOTAL_RUNTIME_S:
        print(f"Time budget reached ({elapsed:.1f}s). Finalizing scorecard.")
        break

    game_id = env_info.game_id
    print(f"[{idx+1}/{len(envs)}] Playing {game_id} (Elapsed: {elapsed:.1f}s)...", flush=True)
    try:
        env = arcade.make(game_id, scorecard_id=card_id)
        if env is None:
            continue
        agent = CohezionGoExploreAgent(game_id, seed=idx * 100)
        obs = env.reset()

        step = 0
        while obs and obs.state not in (GameState.WIN, GameState.GAME_OVER) and step < 150:
            act, data = agent.choose_action(obs)
            obs = env.step(act, data=data)
            step += 1

        print(f"  -> {game_id} finished in {step} steps. State: {obs.state if obs else None}, Levels: {obs.levels_completed if obs else 0}")
    except Exception as exc:
        print(f"  Error on {game_id}: {exc}")

sc = arcade.close_scorecard(card_id)
print(f"✓ Closed Scorecard {card_id}. Total actions: {sc.total_actions if sc else 0}, Completed: {sc.total_levels_completed if sc else 0}")


In [ ]:
# 4. Commit Validator Parquet Handler
# During commit (not TRUE_SUBMISSION), generates initial valid parquet so Kaggle enables Submit.
# During competition rerun (TRUE_SUBMISSION), Kaggle generates submission.parquet automatically from the scorecard.
import os
import pandas as pd
from pathlib import Path

target_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
target_file = target_dir / "submission.parquet"

if not TRUE_SUBMISSION:
    print("Generating commit-time validator submission.parquet...")
    commit_df = pd.DataFrame(
        data=[["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"]
    )
    commit_df.to_parquet(target_file, index=False)
    print(f"✓ Commit validator submission.parquet written ({target_file.stat().st_size} bytes).")
else:
    print("TRUE_SUBMISSION rerun complete. Gateway recorded scorecard for evaluation.")
